##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     `1616714`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## 0 Setup

In [51]:
import pandas as pd
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support
)



categorical_features = [ "workclass", "education", "marital-status", "occupation", "relationship", "race", "sex", "native-country"]
numerical_features = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
ALL_FEATURES = categorical_features + numerical_features
TARGET = "income"

EPS = 1e-9
ALPHA = 1.0

train_df = pd.read_csv('Assignment1_data/adult_supervised_train.csv')
unlabelled_df = pd.read_csv("Assignment1_data/adult_unlabelled.csv")
test_df = pd.read_csv("Assignment1_data/adult_test.csv")

def clean_data(df, concept = True):

    df.replace('?', np.nan, inplace=True)
    if df["fnlwgt"].dtype == "object": 
        df.drop("fnlwgt", inplace = True, axis = 1)
    df.dropna(subset = ALL_FEATURES, inplace = True)
    # encode concept to binary high income or not
    if concept:
        df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

    return df

## 0.5. Helpers

In [52]:
# mean and variances for each class of the continuous features
def gaussian_params(df):
    params = {}
    for column in df['income'].unique():
        subset = df[df['income'] == column]
        params[column] = {
            'mean': subset[numerical_features].mean(),
            'var': subset[numerical_features].var()
        }
    return params

# mean and variance for each class of the categorical features
def categorical_params(df):
    probs = {}
    for column in df['income'].unique():
        subset = df[df['income'] == column]
        probs[column] = {}
        for feature in categorical_features:
            value_counts = subset[feature].value_counts()
            total_count = len(subset)
            probs[column][feature] = (value_counts / total_count).to_dict()
    return probs
    
def gaussian_log_probs(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)



class MixedNaiveBayes:

    def __init__(self):
        # initialize both GaussianNB and CategoricalNB
        self.gnb = GaussianNB()
        self.cnb = CategoricalNB(alpha = ALPHA)

    def fit(self, X_cat, X_cont, y):
        # fit both models separately
        self.gnb.fit(X_cont, y)
        self.cnb.fit(X_cat, y)
        self.classes = self.gnb.classes_

    def predict_log_proba(self, X_cont, X_cat):
        # return combined log probabilities
        log_prob_cont = self.gnb.predict_log_proba(X_cont)
        log_prob_cat = self.cnb.predict_log_proba(X_cat)

        return log_prob_cat + log_prob_cont

    def predict(self, X_cont, X_cat):
        log_probs = self.predict_log_proba(X_cont, X_cat)
        return self.classes[np.argmax(log_probs, axis=1)]

    def posterior_ratio(self, X_cont, X_cat):
        log_probs = self.predict_log_proba(X_cont, X_cat)
        
        return np.exp(log_probs[:, 1] - log_probs[:, 0])




## 1. Supervised model training


Q1.1 Class Priors
Class 0: 1.0000


## 2. Supervised model evaluation

## 3. Extending the model with semi-supervised training

## 4. Supervised model evaluation